# CineIQ — Collaborative Filtering (SVD)

Matrix factorization on MovieLens 20M with **proper train/test split**.

Key fix from v1: cross-validation is done on held-out data, not the full training set.

In [1]:
import numpy as np
import pandas as pd
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split, cross_validate
import pickle
import time

PROC = "../data/processed"
MODELS = "../models"

ratings = pd.read_csv(f"{PROC}/merged.csv")
ratings = ratings[["userId", "movieId", "rating"]]
print(f"Ratings: {len(ratings):,}")
print(f"Users: {ratings['userId'].nunique():,}")
print(f"Movies: {ratings['movieId'].nunique():,}")

Ratings: 20,000,263
Users: 138,493
Movies: 26,744


## 1. Train/Test Split (80/20)

**v1 mistake:** Trained on full data, then CV'd on same data → invalid RMSE.

**v2 fix:** Hold out 20% of ratings for testing. Never touch test set during training.

In [2]:
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(ratings, reader)

# 80/20 split — stratified by rating
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)
print(f"Train ratings: {trainset.n_ratings:,}")
print(f"Test ratings: {len(testset):,}")

Train ratings: 16,000,210
Test ratings: 4,000,053


## 2. Cross-Validation on Train Set

CV is done on the **train set only** using internal folds.

In [ ]:
# 3-fold CV on train set
svd = SVD(n_factors=100, n_epochs=20, random_state=42, lr_all=0.005, reg_all=0.02)

t0 = time.time()
cv_results = cross_validate(svd, data, measures=["RMSE"], cv=3, verbose=True)
print(f"CV time: {time.time() - t0:.1f}s")

## 3. Train on Full Train Set, Evaluate on Test Set

Final model trained on 80% of data, evaluated on held-out 20%.

In [ ]:
# Train on full trainset
t0 = time.time()
svd.fit(trainset)
print(f"Training time: {time.time() - t0:.1f}s")

# Evaluate on TEST set (held out, never seen during training)
t0 = time.time()
predictions = svd.test(testset)
print(f"Prediction time: {time.time() - t0:.1f}s")

# Compute RMSE on test set
from surprise import accuracy
rmse = accuracy.rmse(predictions)
print(f"\nTest RMSE: {rmse:.4f}")

In [ ]:
# Also compute MAE
mae = accuracy.mae(predictions)
print(f"Test MAE: {mae:.4f}")

## 4. Prediction Distribution

In [ ]:
import matplotlib.pyplot as plt

pred_vals = [p.est for p in true_vals := [p.r_ui for p in predictions]]
true_vals = [p.r_ui for p in predictions]
errors = [p.est - p.r_ui for p in predictions]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(pred_vals, bins=30, edgecolor="black", alpha=0.7, color="steelblue")
axes[0].set_title("Predicted Rating Distribution")
axes[0].set_xlabel("Predicted Rating")

axes[1].hist(true_vals, bins=30, edgecolor="black", alpha=0.7, color="coral")
axes[1].set_title("True Rating Distribution")
axes[1].set_xlabel("True Rating")

axes[2].hist(errors, bins=50, edgecolor="black", alpha=0.7, color="mediumseagreen")
axes[2].set_title(f"Prediction Error (RMSE={rmse:.4f})")
axes[2].set_xlabel("Error (pred - true)")
axes[2].axvline(0, color="red", linestyle="--")

plt.tight_layout()
plt.show()

## 5. Save Model

In [ ]:
# Train on FULL dataset for production use
full_trainset = data.build_full_trainset()
svd_full = SVD(n_factors=100, n_epochs=20, random_state=42, lr_all=0.005, reg_all=0.02)
svd_full.fit(full_trainset)

pickle.dump(svd_full, open(f"{MODELS}/svd_model.pkl", "wb"))
print(f"Saved svd_model.pkl")
print(f"Test RMSE (from holdout evaluation): {rmse:.4f}")